# Urban Heat Mitigation AI/ML System
## End-to-End Analysis Notebook

**Problem:** Identify urban heat stress hotspots, quantify heating drivers, and generate optimized cooling interventions using physics-informed AI/ML.

**Pipeline Stages:**
1. Physics-grounded data generation (Landsat LST / ERA5 proxy)
2. UTCI Heat Stress Index computation
3. Getis-Ord Gi* Hotspot Detection
4. Physics-Informed XGBoost Model (energy balance constraints)
5. SHAP Driver Analysis
6. Cooling Scenario Simulation (6 interventions)
7. Budget-constrained Greedy Optimization
8. Visualization

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
os.chdir(os.path.abspath('..'))

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import warnings
warnings.filterwarnings('ignore')

print('Environment ready.')

## Stage 1: Load Configuration & Generate City Data

In [ ]:
from src.data_pipeline.synthetic_data import load_config, generate_city_grid, grid_to_dataframe

config = load_config('config.yaml')
print(f"City: {config['city']['name']}")
print(f"Grid: {config['grid']['rows']}x{config['grid']['cols']} @ {config['grid']['resolution_m']}m resolution")

data = generate_city_grid(config)
df = grid_to_dataframe(data)

print(f"\nDataFrame shape: {df.shape}")
df.describe().round(2)

## Stage 2: Heat Stress Index Computation

In [ ]:
from src.heat_analysis.heat_stress_index import (
    utci_approximation, wbgt_outdoor, classify_heat_stress,
    HEAT_STRESS_LABELS, HEAT_STRESS_COLORS
)

utci = utci_approximation(data['air_temp'], data['humidity'], data['lst'], data['wind_speed'])
wbgt = wbgt_outdoor(data['air_temp'], data['humidity'], data['lst'], data['wind_speed'])
stress_class = classify_heat_stress(utci)

print('Heat Stress Distribution:')
for level in range(6):
    pct = (stress_class == level).mean() * 100
    print(f'  Level {level} [{HEAT_STRESS_LABELS[level]:25s}]: {pct:5.1f}%')

## Stage 3: Hotspot Detection (Getis-Ord Gi*)

In [ ]:
from src.heat_analysis.hotspot_detector import detect_hotspots

hotspot_results = detect_hotspots(data['lst'], utci, config)
hs = hotspot_results['hotspot_stats']

print(f"Total hotspot cells : {hs['n_hotspot_cells']:,} ({hs['hotspot_fraction_pct']}%)")
print(f"Mean LST in hotspots: {hs['mean_lst_hotspot_C']}°C")
print(f"Urban heat excess   : +{hs['lst_excess_C']}°C above city mean")
print(f"Max Gi* Z-score     : {hs['max_gi_star']}")

## Stage 4: Physics-Informed ML Model

In [ ]:
from src.ml_models.piml_model import train_piml_pipeline, predict_full_grid

ml_results = train_piml_pipeline(df, config)
model = ml_results['model']
feature_cols = ml_results['feature_cols']
metrics = ml_results['metrics']

# Predict full grid
lst_predicted_flat = predict_full_grid(model, df, feature_cols)
lst_predicted = lst_predicted_flat.reshape(data['rows'], data['cols'])

print(f"\nModel Metrics:")
for k, v in metrics.items():
    print(f"  {k}: {v}")

## Stage 5: SHAP Driver Analysis

In [ ]:
from src.heat_analysis.driver_analysis import analyze_drivers

driver_results = analyze_drivers(
    model=model,
    X_train=ml_results['X_train'],
    feature_cols=feature_cols,
    n_shap_samples=2000,
)

print('\nTop 8 Urban Heat Drivers:')
print(driver_results['importance_df'].head(8)[['rank','label','pct_contribution']].to_string(index=False))

print('\nDriver Category Breakdown:')
print(driver_results['grouped_df'].to_string(index=False))

## Stage 6: Cooling Scenario Simulation

In [ ]:
from src.cooling_scenarios.scenario_engine import run_all_scenarios

scenario_results = run_all_scenarios(
    df_baseline=df,
    lst_baseline=lst_predicted,
    model=model,
    feature_cols=feature_cols,
    hotspot_class=hotspot_results['hotspot_class'],
    lulc_grid=data['lulc'],
    config=config,
)

print('\nScenario Summary:')
print(scenario_results['summary_df'].to_string(index=False))

## Stage 7: Budget-Constrained Optimization

In [ ]:
from src.cooling_scenarios.optimizer import greedy_optimize

opt_results = greedy_optimize(
    df_baseline=df,
    lst_baseline=lst_predicted,
    model=model,
    feature_cols=feature_cols,
    hotspot_class=hotspot_results['hotspot_class'],
    lulc_grid=data['lulc'],
    config=config,
)

print(f"\nOptimal Strategy Results:")
print(f"  Budget used    : {opt_results['budget_used']}/{opt_results['budget_total']} units")
print(f"  Area covered   : {opt_results['area_covered_km2']} km2")
print(f"  Mean cooling   : -{opt_results['mean_cooling_hotspots_C']}C in hotspots")
print(f"  City-wide mean : -{opt_results['mean_cooling_city_C']}C")
print(f"  Mix            : {opt_results['intervention_allocation']}")

## Stage 8: Generate All Visualizations

In [ ]:
from src.visualization.heat_maps import (
    plot_lulc_map, plot_lst_map, plot_heat_stress_map,
    plot_hotspot_map, plot_optimal_strategy_map
)
from src.visualization.driver_plots import (
    plot_shap_summary, plot_driver_importance_bar,
    plot_driver_category_pie, plot_scenario_comparison,
    plot_model_validation
)

os.makedirs('outputs/maps', exist_ok=True)
os.makedirs('outputs/figures', exist_ok=True)

plot_lulc_map(data['lulc'], config, 'outputs/maps')
plot_lst_map(data['lst'], config, 'outputs/maps', title_suffix=' (Baseline)')
plot_heat_stress_map(stress_class, config, 'outputs/maps')
plot_hotspot_map(hotspot_results['hotspot_class'], hotspot_results['gi_star_lst'], config, 'outputs/maps')
plot_optimal_strategy_map(lst_predicted, opt_results['lst_optimal'], opt_results['delta_optimal'],
                          opt_results['allocation_map'], config, 'outputs/maps')

plot_model_validation(ml_results['y_test'].values, ml_results['y_pred_test'], metrics, 'outputs/figures')
plot_shap_summary(driver_results['shap_values'], driver_results['X_sample'], feature_cols, 'outputs/figures')
plot_driver_importance_bar(driver_results['importance_df'], 'outputs/figures')
plot_driver_category_pie(driver_results['grouped_df'], 'outputs/figures')
plot_scenario_comparison(scenario_results['scenarios'], 'outputs/figures')

print('All visualizations saved!')

## View Output Maps

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(22, 13))
map_files = [
    ('outputs/maps/lulc_map.png', 'LULC Map'),
    ('outputs/maps/lst_baseline.png', 'Land Surface Temperature'),
    ('outputs/maps/heat_stress_map.png', 'Heat Stress (UTCI)'),
    ('outputs/maps/hotspot_map.png', 'Hotspot Detection (Gi*)'),
    ('outputs/figures/shap_summary.png', 'SHAP Driver Analysis'),
    ('outputs/maps/optimal_strategy_map.png', 'Optimal Cooling Strategy'),
]
for ax, (fpath, title) in zip(axes.ravel(), map_files):
    if os.path.exists(fpath):
        img = mpimg.imread(fpath)
        ax.imshow(img)
        ax.set_title(title, fontsize=12, fontweight='bold')
    ax.axis('off')

plt.suptitle('Urban Heat Mitigation — Results Dashboard', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('outputs/figures/dashboard.png', dpi=120, bbox_inches='tight')
plt.show()
print('Dashboard saved.')